In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
import copy
warnings.filterwarnings('ignore')

# 🚀 GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스 상태: {device} (딥러닝 가속 중)")

# ==========================================
# [공통] 정형 데이터 전용 심층 신경망(MLP)
# ==========================================
class TabularDNN(nn.Module):
    def __init__(self, input_dim):
        super(TabularDNN, self).__init__()
        # [핵심] BatchNorm 대신 LayerNorm을 사용하여 배치 크기에 따른 NaN 붕괴 원천 차단!
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.Mish(),
            nn.Dropout(0.3),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.Mish(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.Mish(),
            
            nn.Linear(64, 1) # 지연 시간 예측 (회귀)
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 1. 완전체 전처리 및 K-Means 군집화
# ==========================================
def preprocess_and_cluster_dl(train_path, layout_path):
    print("\n1. 데이터 로드 및 '완전체' 심화 전처리 시작...")
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col].values
    X_train = train.drop(target_col, axis=1)
    
    # 결측치 처리
    for col in X_train.select_dtypes(include=['float64', 'int64']).columns:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    # 파생 변수 (주기성 원본 삭제)
    if 'shift_hour' in X_train.columns:
        X_train['shift_hour_sin'] = np.sin(2 * np.pi * X_train['shift_hour'] / 24)
        X_train['shift_hour_cos'] = np.cos(2 * np.pi * X_train['shift_hour'] / 24)
    if 'day_of_week' in X_train.columns:
        X_train['day_of_week_sin'] = np.sin(2 * np.pi * X_train['day_of_week'] / 7)
        X_train['day_of_week_cos'] = np.cos(2 * np.pi * X_train['day_of_week'] / 7)
    X_train.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True, errors='ignore') 
    
    # 파생 변수 (상호작용)
    # 여기 끼워넣기
    if 'robot_utilization' in X_train.columns and 'congestion_score' in X_train.columns:
        X_train['robot_util_x_congestion'] = X_train['robot_utilization'] * X_train['congestion_score']
    if 'order_inflow_15m' in X_train.columns and 'robot_active' in X_train.columns:
        X_train['order_pressure'] = X_train['order_inflow_15m'] / (X_train['robot_active'] + 1)
        
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)

    print("2. 창고 상태(쾌적/혼잡/마비) 3분할 군집화 진행 중...")
    possible_features = ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m']
    cluster_features = [f for f in possible_features if f in X_train.columns]
    
    X_cluster = X_train[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    scaler_cluster = StandardScaler()
    X_scaled_cluster = scaler_cluster.fit_transform(X_cluster)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled_cluster)
    
    # [방어 로직] 파생 변수 생성 중 발생할 수 있는 무한대(inf)나 결측치를 최종 0으로 치환
    X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train.fillna(0, inplace=True)
    
    return X_train, y_train

# ==========================================
# 2. 오직 딥러닝(DNN)만 사용하는 군집별 분할 학습
# ==========================================
def validate_dl_only_stable(X_train_df, y_train_np):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_predictions = np.zeros(len(X_train_df))
    
    print("\n3. 군집별 '순수 딥러닝(PyTorch)' 5-Fold 검증 시작...")
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_df)):
        print(f"\n--- Fold {fold+1} 시작 ---")
        x_tr = X_train_df.iloc[train_idx].copy()
        y_tr = y_train_np[train_idx]
        x_val = X_train_df.iloc[val_idx].copy()
        y_val = y_train_np[val_idx]
        
        overall_mean = y_tr.mean()
        
        # Target Encoding
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                y_tr_series = pd.Series(y_tr, index=x_tr.index)
                group_stats = y_tr_series.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                
                x_tr[f'{col}_target_enc'] = x_tr[col].map(smoothed_means)
                x_val[f'{col}_target_enc'] = x_val[col].map(smoothed_means).fillna(overall_mean)
                x_tr.drop(col, axis=1, inplace=True)
                x_val.drop(col, axis=1, inplace=True)

        fold_preds = np.zeros(len(x_val))
        
        # 3개의 군집별로 각기 다른 딥러닝 모델(DNN) 학습
        for cid in range(3):
            tr_mask = (x_tr['cluster_id'] == cid).values
            val_mask = (x_val['cluster_id'] == cid).values
            
            if not val_mask.any() or not tr_mask.any(): continue
            
            x_tr_c = x_tr[tr_mask].drop('cluster_id', axis=1)
            y_tr_c = y_tr[tr_mask]
            x_val_c = x_val[val_mask].drop('cluster_id', axis=1)
            y_val_c = y_val[val_mask]
            
            # 딥러닝 필수: 군집별 스케일링
            scaler = StandardScaler()
            x_tr_scaled = scaler.fit_transform(x_tr_c)
            x_val_scaled = scaler.transform(x_val_c)
            
            train_dataset = TensorDataset(torch.FloatTensor(x_tr_scaled), torch.FloatTensor(y_tr_c).unsqueeze(1))
            val_dataset = TensorDataset(torch.FloatTensor(x_val_scaled), torch.FloatTensor(y_val_c).unsqueeze(1))
            
            train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
            
            model = TabularDNN(input_dim=x_tr_c.shape[1]).to(device)
            criterion = nn.L1Loss() # MAE
            # 학습률을 0.001로 낮추어 안정성 확보
            optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
            
            best_val_loss = float('inf')
            patience_counter = 0
            # [방어 로직] 훈련 시작 전 초기 가중치 저장 (NoneType 에러 방지)
            best_model_weights = copy.deepcopy(model.state_dict())
            
            for epoch in range(100):
                model.train()
                for inputs, targets in train_loader:
                    inputs, targets = inputs.to(device), targets.to(device)
                    optimizer.zero_grad()
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    loss.backward()
                    
                    # [방어 로직] 기울기 폭발(NaN) 방지를 위한 클리핑
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                model.eval()
                val_loss = 0.0
                with torch.no_grad():
                    for inputs, targets in val_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        val_loss += criterion(model(inputs), targets).item() * inputs.size(0)
                val_loss /= len(val_loader.dataset)
                
                scheduler.step(val_loss)
                
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    # [방어 로직] 확실한 깊은 복사
                    best_model_weights = copy.deepcopy(model.state_dict())
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= 15: # Early Stopping
                        break
            
            # 검증용 예측
            model.load_state_dict(best_model_weights)
            model.eval()
            pred_dnn = []
            with torch.no_grad():
                for inputs, _ in val_loader:
                    inputs = inputs.to(device)
                    pred_dnn.extend(model(inputs).cpu().numpy().flatten())
            
            fold_preds[val_mask] = np.array(pred_dnn)
            
        oof_predictions[val_idx] = fold_preds
        print(f"  > Fold {fold+1} 딥러닝 MAE: {mean_absolute_error(y_val, fold_preds):.4f}")
        
    total_mae = mean_absolute_error(y_train_np, oof_predictions)
    print("\n" + "="*50)
    print(f"🔥 [오직 딥러닝] 분할정복 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")

if __name__ == "__main__":
    TRAIN_PATH = 'train.csv' # 실제 환경에 맞게 파일명 확인
    LAYOUT_PATH = 'layout_info.csv'
    
    X_train_df, y_train_np = preprocess_and_cluster_dl(TRAIN_PATH, LAYOUT_PATH)
    validate_dl_only_stable(X_train_df, y_train_np)

디바이스 상태: cpu (딥러닝 가속 중)

1. 데이터 로드 및 '완전체' 심화 전처리 시작...
2. 창고 상태(쾌적/혼잡/마비) 3분할 군집화 진행 중...

3. 군집별 '순수 딥러닝(PyTorch)' 5-Fold 검증 시작...

--- Fold 1 시작 ---
  > Fold 1 딥러닝 MAE: 4.9863

--- Fold 2 시작 ---


KeyboardInterrupt: 